# Question Paper Generator (MCQ + Essay, Bloom's Taxonomy) - Gemini Flash edition

Same pipeline as before, but uses **Google Gemini Flash** for question generation instead of the Claude API, so you can run this entirely on Google's free tier.

Get a free API key at: https://aistudio.google.com/apikey (no billing required for the free tier, with rate limits).

Pipeline:
1. Upload one or more lesson-note PDFs (OCR + direct text extraction)
2. Fill in assessment details (title, board, level, subject, MCQ/Essay counts, Bloom's %)
3. Allocate questions across Bloom's levels, then split each level's count into MCQ/Essay
4. Generate questions per (topic, Bloom's level, type) using Gemini Flash
5. Assemble: **Section A - MCQ** (easy -> hard), **Section B - Essay** (easy -> hard), then an **Answer Key** page
6. Export as a formatted `.docx` file

## 1. Install dependencies

In [1]:
!apt-get install -y tesseract-ocr -q
!pip install pymupdf pytesseract pillow python-docx google-genai -q

Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 18.1 MB/s eta 0:00:00


In [2]:
import fitz  # PyMuPDF
import pytesseract
from PIL import Image, ImageOps
import io, json, re, getpass, time
from google.colab import files
from google import genai
from google.genai import types
from docx import Document
from docx.shared import Pt, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH

## 2. Gemini API key

Get a free key at https://aistudio.google.com/apikey. Entered via a hidden prompt, not stored in the notebook.

In [3]:
API_KEY = getpass.getpass("Enter your Gemini API key: ")
client = genai.Client(api_key=API_KEY)
MODEL = "gemini-3.5-flash"  # free-tier friendly; swap to 'gemini-2.0-flash-lite' if you hit rate limits

Enter your Gemini API key: ··········


## 3. Upload lesson note PDFs

Upload one PDF per lesson/topic. You'll be asked to label each one (e.g. "Forces & Motion").

In [4]:
uploaded = files.upload()
pdf_paths = list(uploaded.keys())
print(f"Uploaded {len(pdf_paths)} file(s): {pdf_paths}")

Saving 1474600428.pdf to 1474600428.pdf
Saving Psychology full -syllabus.pdf to Psychology full -syllabus.pdf
Uploaded 2 file(s): ['1474600428.pdf', 'Psychology full -syllabus.pdf']


In [5]:
topic_labels = {}
for p in pdf_paths:
    label = input(f"Enter a topic label for '{p}' (e.g. 'Forces & Motion'): ").strip()
    topic_labels[p] = label if label else p

Enter a topic label for '1474600428.pdf' (e.g. 'Forces & Motion'): Psychology Unit 2
Enter a topic label for 'Psychology full -syllabus.pdf' (e.g. 'Forces & Motion'): Psychology Unit 1


## 4. Extraction (direct text + OCR fallback, per page)

In [6]:
TEXT_LENGTH_THRESHOLD = 20
RENDER_DPI = 300
OCR_LANG = 'eng'

def render_page_to_image(page, dpi=300):
    zoom = dpi / 72
    pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom))
    return Image.open(io.BytesIO(pix.tobytes("png")))

def preprocess_for_ocr(img):
    img = img.convert('L')
    return ImageOps.autocontrast(img)

def ocr_page(page, dpi=300, lang='eng'):
    img = preprocess_for_ocr(render_page_to_image(page, dpi=dpi))
    return pytesseract.image_to_string(img, lang=lang).strip()

def extract_pdf_text(pdf_path, text_threshold=20, dpi=300, lang='eng'):
    doc = fitz.open(pdf_path)
    pages = []
    for page_num in range(len(doc)):
        page = doc[page_num]
        direct_text = page.get_text().strip()
        if len(direct_text) >= text_threshold:
            pages.append(direct_text)
        else:
            pages.append(ocr_page(page, dpi=dpi, lang=lang))
    doc.close()
    return "\n".join(pages)

extracted_docs = []
for p in pdf_paths:
    print(f"Extracting: {p} ...")
    text = extract_pdf_text(p, TEXT_LENGTH_THRESHOLD, RENDER_DPI, OCR_LANG)
    extracted_docs.append({"topic_label": topic_labels[p], "text": text})
    print(f"  -> {len(text)} characters extracted\n")

print("Extraction complete.")

Extracting: 1474600428.pdf ...
  -> 24544 characters extracted

Extracting: Psychology full -syllabus.pdf ...
  -> 125166 characters extracted

Extraction complete.


## 5. Assessment details (form inputs)

In [7]:
assessment_title = input("Assessment Title: ").strip()

print("\nExam Board options: 1) Cambridge  2) Edexcel")
board_choice = input("Select Exam Board (1/2): ").strip()
exam_board = "Cambridge" if board_choice == "1" else "Edexcel"

CAMBRIDGE_LEVELS = ["Cambridge Primary", "Cambridge Lower Secondary", "Cambridge IGCSE", "Cambridge International AS & A Level"]
EDEXCEL_LEVELS = ["Edexcel International Primary", "Edexcel International Lower Secondary", "Edexcel International GCSE", "Edexcel International A Level"]
levels = CAMBRIDGE_LEVELS if exam_board == "Cambridge" else EDEXCEL_LEVELS

print(f"\nQualification Level options for {exam_board}:")
for i, lvl in enumerate(levels, 1):
    print(f"  {i}) {lvl}")
level_choice = int(input("Select Qualification Level (number): ").strip())
qualification_level = levels[level_choice - 1]

SUBJECTS = ["Physics", "Chemistry", "Biology", "Mathematics", "Computer Science"]
print("\nSubject options:")
for i, s in enumerate(SUBJECTS, 1):
    print(f"  {i}) {s}")
subject_choice = int(input("Select Subject (number): ").strip())
subject = SUBJECTS[subject_choice - 1]

num_mcq = int(input("\nNumber of MCQ questions: ").strip())
num_essay = int(input("Number of Essay questions: ").strip())
num_questions = num_mcq + num_essay

print(f"\nEnter Bloom's Taxonomy percentages (must sum to 100). Total questions = {num_questions}")
blooms_distribution = {}
for level in ["remember", "understand", "apply", "analyze", "evaluate", "create"]:
    val = float(input(f"  {level.capitalize()} %: ").strip())
    blooms_distribution[level] = val

assert abs(sum(blooms_distribution.values()) - 100) < 0.01, "Bloom's percentages must sum to 100"

print("\n--- Summary ---")
print(f"Title: {assessment_title}")
print(f"Board: {exam_board} | Level: {qualification_level} | Subject: {subject}")
print(f"MCQ: {num_mcq}  Essay: {num_essay}  Total: {num_questions}")
print(f"Bloom's distribution: {blooms_distribution}")

Assessment Title: Test 2

Exam Board options: 1) Cambridge  2) Edexcel
Select Exam Board (1/2): 2

Qualification Level options for Edexcel:
  1) Edexcel International Primary
  2) Edexcel International Lower Secondary
  3) Edexcel International GCSE
  4) Edexcel International A Level
Select Qualification Level (number): 3

Subject options:
  1) Physics
  2) Chemistry
  3) Biology
  4) Mathematics
  5) Computer Science
Select Subject (number): 3

Number of MCQ questions: 25
Number of Essay questions: 6

Enter Bloom's Taxonomy percentages (must sum to 100). Total questions = 31
  Remember %: 10
  Understand %: 20
  Apply %: 20
  Analyze %: 20
  Evaluate %: 20
  Create %: 10

--- Summary ---
Title: Test 2
Board: Edexcel | Level: Edexcel International GCSE | Subject: Biology
MCQ: 25  Essay: 6  Total: 31
Bloom's distribution: {'remember': 10.0, 'understand': 20.0, 'apply': 20.0, 'analyze': 20.0, 'evaluate': 20.0, 'create': 10.0}


## 6. Allocation logic

Two steps:
1. Convert Bloom's percentages into exact integer counts summing to `num_questions`.
2. Split each Bloom's level's count into MCQ vs Essay, based on an affinity table (e.g. 'create' is essay-only, 'remember' is mostly MCQ), while respecting the exact `num_mcq`/`num_essay` totals.

In [8]:
BLOOMS_ORDER = ["remember", "understand", "apply", "analyze", "evaluate", "create"]

BLOOMS_TYPE_AFFINITY = {
    "remember":   {"mcq": 1.0, "essay": 0.1},
    "understand": {"mcq": 0.8, "essay": 0.3},
    "apply":      {"mcq": 0.6, "essay": 0.5},
    "analyze":    {"mcq": 0.3, "essay": 0.8},
    "evaluate":   {"mcq": 0.1, "essay": 1.0},
    "create":     {"mcq": 0.0, "essay": 1.0},
}

BLOOMS_COMMAND_WORDS = {
    "remember": "State, Define, List, Identify, Name",
    "understand": "Describe, Explain, Summarize, Outline",
    "apply": "Calculate, Demonstrate, Solve, Use",
    "analyze": "Compare, Contrast, Examine, Differentiate",
    "evaluate": "Justify, Assess, Critique, Evaluate",
    "create": "Design, Propose, Formulate, Construct",
}

def allocate_questions_by_blooms(num_questions, blooms_distribution):
    raw = {lvl: (pct / 100) * num_questions for lvl, pct in blooms_distribution.items()}
    floored = {lvl: int(v) for lvl, v in raw.items()}
    remainder = num_questions - sum(floored.values())
    fractions = sorted(raw.items(), key=lambda x: x[1] - int(x[1]), reverse=True)
    for i in range(remainder):
        floored[fractions[i][0]] += 1
    return floored

def allocate_type_by_blooms(blooms_counts, num_mcq, num_essay, affinity=BLOOMS_TYPE_AFFINITY):
    remaining_mcq, remaining_essay = num_mcq, num_essay
    allocation = {lvl: {"mcq": 0, "essay": 0} for lvl in blooms_counts}
    ordered_levels = sorted(
        blooms_counts.keys(),
        key=lambda lvl: abs(affinity[lvl]["mcq"] - affinity[lvl]["essay"]),
        reverse=True
    )
    for level in ordered_levels:
        count = blooms_counts[level]
        mcq_w, essay_w = affinity[level]["mcq"], affinity[level]["essay"]
        mcq_share = 0 if (mcq_w + essay_w == 0) else round(count * mcq_w / (mcq_w + essay_w))
        mcq_share = min(mcq_share, remaining_mcq, count)
        essay_share = min(count - mcq_share, remaining_essay)
        leftover = count - mcq_share - essay_share
        if leftover > 0:
            extra_mcq = min(leftover, remaining_mcq - mcq_share)
            mcq_share += extra_mcq
            leftover -= extra_mcq
        if leftover > 0:
            extra_essay = min(leftover, remaining_essay - essay_share)
            essay_share += extra_essay
        allocation[level]["mcq"] = mcq_share
        allocation[level]["essay"] = essay_share
        remaining_mcq -= mcq_share
        remaining_essay -= essay_share
    return allocation

def allocate_across_topics(count, topic_labels_list):
    if count == 0 or not topic_labels_list:
        return {t: 0 for t in topic_labels_list}
    n = len(topic_labels_list)
    base, remainder = divmod(count, n)
    alloc = {t: base for t in topic_labels_list}
    for t in topic_labels_list[:remainder]:
        alloc[t] += 1
    return alloc

blooms_counts = allocate_questions_by_blooms(num_questions, blooms_distribution)
type_allocation = allocate_type_by_blooms(blooms_counts, num_mcq, num_essay)

print("Bloom's level counts:", blooms_counts)
print("Type split per level:", type_allocation)

if blooms_counts.get("create", 0) > 0 and num_essay == 0:
    print("\nWARNING: 'Create' level needs essay format but essay count is 0. Consider adjusting inputs.")

Bloom's level counts: {'remember': 3, 'understand': 7, 'apply': 6, 'analyze': 6, 'evaluate': 6, 'create': 3}
Type split per level: {'remember': {'mcq': 3, 'essay': 0}, 'understand': {'mcq': 7, 'essay': 0}, 'apply': {'mcq': 6, 'essay': 0}, 'analyze': {'mcq': 6, 'essay': 0}, 'evaluate': {'mcq': 3, 'essay': 3}, 'create': {'mcq': 0, 'essay': 3}}


## 7. Generate questions via Gemini Flash

Uses Gemini's structured output mode (`response_mime_type: application/json`) so you get clean JSON back without needing to strip markdown fences.

A small delay + retry is included since the free tier has a requests-per-minute limit (check current limits at https://ai.google.dev/gemini-api/docs/rate-limits).

In [9]:
REQUEST_DELAY_SECONDS = 2  # be polite to the free-tier rate limit; raise this if you hit 429 errors

def call_gemini_for_questions(topic_text, topic_label, blooms_level, qtype, n, subject, exam_board, qualification_level, max_retries=3):
    command_words = BLOOMS_COMMAND_WORDS[blooms_level]

    if qtype == "mcq":
        format_instructions = (
            "Each question must have exactly 4 options (A, B, C, D), one correct answer. "
            "Distractors should be plausible, based on common misconceptions from the source material. "
            "Return a JSON list, each item shaped like: "
            '{"question": str, "options": {"A": str, "B": str, "C": str, "D": str}, "correct_answer": "A"|"B"|"C"|"D", "marks": int}'
        )
    else:
        format_instructions = (
            "Each question requires an extended written response, not a single fact. "
            "Include a mark allocation and a marking scheme (list of expected discussion points). "
            "Return a JSON list, each item shaped like: "
            '{"question": str, "marks": int, "marking_scheme": [str, str, ...]}'
        )

    prompt = f"""You are an exam question writer for {exam_board} {qualification_level} {subject}.

Using ONLY the following lesson content (topic: \"{topic_label}\") as source material, write {n} exam question(s) at the \"{blooms_level}\" level of Bloom's Taxonomy.

Use command words appropriate to this level: {command_words}.

{format_instructions}

Return ONLY the JSON list. No markdown fences, no preamble, no explanation.

Lesson content:
\"\"\"
{topic_text[:12000]}
\"\"\"
"""

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    temperature=0.7,
                )
            )
            time.sleep(REQUEST_DELAY_SECONDS)
            cleaned = re.sub(r"```json|```", "", response.text).strip()
            questions = json.loads(cleaned)
            for q in questions:
                q["type"] = qtype
                q["blooms_level"] = blooms_level
                q["topic"] = topic_label
            return questions
        except json.JSONDecodeError:
            print(f"  WARNING: failed to parse JSON for {topic_label}/{blooms_level}/{qtype} (attempt {attempt+1}). Retrying...")
            time.sleep(REQUEST_DELAY_SECONDS)
        except Exception as e:
            print(f"  WARNING: request error for {topic_label}/{blooms_level}/{qtype} (attempt {attempt+1}): {e}")
            time.sleep(REQUEST_DELAY_SECONDS * 3)  # back off more on rate-limit / API errors

    print(f"  FAILED after {max_retries} attempts: {topic_label}/{blooms_level}/{qtype}. Skipping.")
    return []

In [11]:
topic_names = [d["topic_label"] for d in extracted_docs]
topic_text_lookup = {d["topic_label"]: d["text"] for d in extracted_docs}

all_questions = []

for blooms_level in BLOOMS_ORDER:
    for qtype in ["mcq", "essay"]:
        level_type_count = type_allocation[blooms_level][qtype]
        if level_type_count == 0:
            continue
        topic_split = allocate_across_topics(level_type_count, topic_names)
        for topic, n in topic_split.items():
            if n == 0:
                continue
            print(f"Generating {n} {qtype.upper()} question(s) | {blooms_level} | topic: {topic}")
            qs = call_gemini_for_questions(
                topic_text_lookup[topic], topic, blooms_level, qtype, n,
                subject, exam_board, qualification_level
            )
            all_questions.extend(qs)

print(f"\nTotal questions generated: {len(all_questions)} (target was {num_questions})")

Generating 2 MCQ question(s) | remember | topic: Psychology Unit 2
Generating 1 MCQ question(s) | remember | topic: Psychology Unit 1
Generating 4 MCQ question(s) | understand | topic: Psychology Unit 2
Generating 3 MCQ question(s) | understand | topic: Psychology Unit 1
Generating 3 MCQ question(s) | apply | topic: Psychology Unit 2
Generating 3 MCQ question(s) | apply | topic: Psychology Unit 1
Generating 3 MCQ question(s) | analyze | topic: Psychology Unit 2
Generating 3 MCQ question(s) | analyze | topic: Psychology Unit 1
Generating 2 MCQ question(s) | evaluate | topic: Psychology Unit 2
Generating 1 MCQ question(s) | evaluate | topic: Psychology Unit 1
Generating 2 ESSAY question(s) | evaluate | topic: Psychology Unit 2
Generating 1 ESSAY question(s) | evaluate | topic: Psychology Unit 1
Generating 2 ESSAY question(s) | create | topic: Psychology Unit 2
  FAILED after 3 attempts: Psychology Unit 2/create/essay. Skipping.
Generating 1 ESSAY question(s) | create | topic: Psychology 

## 8. Order questions: Section A (MCQ, easy->hard), Section B (Essay, easy->hard)

In [12]:
blooms_rank = {lvl: i for i, lvl in enumerate(BLOOMS_ORDER)}

mcq_questions = sorted(
    [q for q in all_questions if q["type"] == "mcq"],
    key=lambda q: blooms_rank[q["blooms_level"]]
)
essay_questions = sorted(
    [q for q in all_questions if q["type"] == "essay"],
    key=lambda q: blooms_rank[q["blooms_level"]]
)

total_marks = sum(q.get("marks", 0) for q in mcq_questions + essay_questions)
print(f"MCQ: {len(mcq_questions)}  Essay: {len(essay_questions)}  Total marks: {total_marks}")

MCQ: 25  Essay: 3  Total marks: 64


## 9. Render to DOCX

In [13]:
def build_docx(filename, header, mcq_questions, essay_questions, total_marks):
    doc = Document()

    # --- Header ---
    title_p = doc.add_paragraph()
    title_p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = title_p.add_run(header["title"])
    run.bold = True
    run.font.size = Pt(18)

    meta_p = doc.add_paragraph()
    meta_p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    meta_line = f"{header['exam_board']}  |  {header['qualification_level']}  |  {header['subject']}"
    meta_p.add_run(meta_line).italic = True

    info_p = doc.add_paragraph()
    info_p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    info_p.add_run(f"Total Questions: {len(mcq_questions) + len(essay_questions)}   |   Total Marks: {total_marks}")

    name_p = doc.add_paragraph()
    name_p.add_run("Name: ____________________________        Date: ______________")

    doc.add_paragraph("_" * 90)

    # --- Section A: MCQ ---
    if mcq_questions:
        doc.add_heading("Section A: Multiple Choice Questions", level=1)
        for i, q in enumerate(mcq_questions, 1):
            p = doc.add_paragraph()
            p.add_run(f"{i}. {q['question']}  ")
            p.add_run(f"[{q.get('marks', 1)} mark(s)]").italic = True
            for letter in ["A", "B", "C", "D"]:
                opt_text = q.get("options", {}).get(letter, "")
                doc.add_paragraph(f"    {letter}) {opt_text}")
            doc.add_paragraph()

    # --- Section B: Essay ---
    if essay_questions:
        doc.add_heading("Section B: Essay Questions", level=1)
        for i, q in enumerate(essay_questions, 1):
            p = doc.add_paragraph()
            p.add_run(f"{i}. {q['question']}  ")
            p.add_run(f"[{q.get('marks', 0)} mark(s)]").italic = True
            doc.add_paragraph()

    # --- Page break before Answer Key ---
    doc.add_page_break()
    doc.add_heading("Answer Key", level=1)

    if mcq_questions:
        doc.add_heading("Section A - MCQ Answers", level=2)
        for i, q in enumerate(mcq_questions, 1):
            doc.add_paragraph(f"{i}. {q.get('correct_answer', '?')}")

    if essay_questions:
        doc.add_heading("Section B - Essay Marking Schemes", level=2)
        for i, q in enumerate(essay_questions, 1):
            doc.add_paragraph(f"{i}. {q['question']} [{q.get('marks', 0)} marks]").runs[0].bold = True
            for point in q.get("marking_scheme", []):
                doc.add_paragraph(f"    - {point}")
            doc.add_paragraph()

    doc.save(filename)
    return filename

header = {
    "title": assessment_title,
    "exam_board": exam_board,
    "qualification_level": qualification_level,
    "subject": subject,
}

output_filename = re.sub(r'[^A-Za-z0-9]+', '_', assessment_title).strip('_') + ".docx"
build_docx(output_filename, header, mcq_questions, essay_questions, total_marks)
print(f"Saved: {output_filename}")

Saved: Test_2.docx


## 10. Download the question paper

In [14]:
files.download(output_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>